In [1]:
import time
import logging
from typing import Tuple, Dict
from collections import namedtuple
import copy
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree
from scipy.stats import norm
from matplotlib import pyplot as plt
from sklearn import tree
from scipy.stats import ks_2samp

from tqdm import tqdm

from common import *
from common import _get_density_ratio_from_classifier
from data_loader import DataLoader
from estimate_datashifter import DetectorTestExplainer, OUTCOME_MODEL_ARGS
from decomp_explainer import InferenceResult, ExplainerInference
from common import RevisedModel

SPLIT_RATIO = 0.5

In [17]:
def get_bootstrap_metric(Y_true, Y_pred, Y_prob, n_bootstrap=1000, alpha=0.05, rng_seed=0):
    """
    Bootstrap estimate of accuracy and AUC
    """
    rng = np.random.RandomState(rng_seed)
    indices = rng.randint(0, len(Y_true), (n_bootstrap, len(Y_true)))
    Y_true_boot = Y_true[indices]
    Y_pred_boot = Y_pred[indices]
    Y_prob_boot = Y_prob[indices]
    
    acc = np.mean(Y_true_boot == Y_pred_boot, axis=1)
    auc = np.array([roc_auc_score(Y_true_boot[i], Y_prob_boot[i]) for i in range(n_bootstrap)])
    
    results = {
        'acc': np.quantile(acc, q=[alpha/2, 0.5, 1-alpha/2]),
        'auc': np.quantile(auc, q=[alpha/2, 0.5, 1-alpha/2])
    }
    return results

def get_acc_auc(model, X, Y):
    Y = Y.flatten().astype(int)
    pred_prob = model.predict_proba(X)[:,1]
    pred_Y = model.predict(X).flatten().astype(int)
    assert len(X)==len(Y), "X, Y should be of same shape"
    conf_intervals = get_bootstrap_metric(Y, pred_Y, pred_prob, n_bootstrap=1000, alpha=0.05)  # format is (alpha/2, 0.5, 1-alpha/2) quartiles
    performances = {
        'acc': [(pred_Y == Y).mean()],
        'acc_lower': [conf_intervals['acc'][0]],
        'acc_upper': [conf_intervals['acc'][2]],
        'auc': [roc_auc_score(Y, pred_prob)],
        'auc_lower': [conf_intervals['auc'][0]],
        'auc_upper': [conf_intervals['auc'][2]],
    }  # each value is a list so it in can be converted to dataframe
    return performances

def get_covariate_detector(explainer_data, X=None):
    with open(explainer_data, 'rb') as f:
        detectors = pickle.load(f)
        detector, omega = detectors['agg_detectors_x'][0]
    if X is not None:
        y = detector.predict(X, omega)
        mdl = GradientBoostingRegressor(max_depth=2, n_estimators=200)
        mdl.fit(X, y)
        tree_importance = mdl.feature_importances_
        print("Tree importance per feature\n %s" % sorted(list(enumerate(tree_importance)), key = lambda x: x[1], reverse=True))
    return detector, tree_importance

def get_outcome_detector(explainer_data):
    with open(explainer_data, 'rb') as f:
        detectors = pickle.load(f)
        detectors_y = detectors['agg_detectors_y'][0]
    tree_importance = detectors_y.feature_importances_
    print("Tree importance per feature\n %s" % sorted(list(enumerate(tree_importance)), key = lambda x: x[1], reverse=True))
    return detectors_y, tree_importance

In [84]:
explainer_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_linear_aggbiggergroupratioeighty/accuracy/0.0/40/2000/explainerJOB.pkl'
num_jobs = 50
loss = 'accuracy'
decomposition = 'Cond_Outcome'
tolerance = 0.0
grouping_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_grouping_dim4_norm1.csv'
train_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg_output/dim10_linear_aggbiggergroupratioeighty/source_train_data1.csv'
source_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_linear_aggbiggergroupratioeighty/2000/source_dataJOB.csv'
target_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_linear_aggbiggergroupratioeighty/2000/target_dataJOB.csv'
mdl_file = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_linear_aggbiggergroupratioeighty/mdl.pkl'

In [93]:
explainer_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_norm2_aggsubgroupsmallcoeff/accuracy/0.0/40/2000/explainerJOB.pkl'
num_jobs = 50
loss = 'accuracy'
decomposition = 'Cond_Cov'
tolerance = 0.0
grouping_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation/feature_grouping_dim4_norm1.csv'
train_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg_output/dim10_norm2_aggsubgroupsmallcoeff/source_train_data1.csv'
source_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_norm2_aggsubgroupsmallcoeff/2000/source_dataJOB.csv'
target_data = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_norm2_aggsubgroupsmallcoeff/2000/target_dataJOB.csv'
mdl_file = '/Users/harvineetsingh/Documents/lab/perf_diff_attr/explain_score_test/src/simulation_agg/_output/dim10_norm2_aggsubgroupsmallcoeff/mdl.pkl'

## Overlap in detected group for aggregate setup

In [5]:
def get_subgroup_recall(explainer_data, target_data):
    targetX, targetY = read_csv(target_data)
    targetX = targetX.to_numpy()

    detector = get_outcome_detector(explainer_data)
    detections_target = detector.predict(targetX) > 0

    # true_subgroup = (targetX[:,0] < -3.5) | (targetX[:,0] > 3.5)  # outcome
    true_subgroup = (targetX[:,0] < -4) | (targetX[:,0] > 4)  # covariate
    intersecting_subgroup = true_subgroup & detections_target
    print("True subgroup size %.2f from %d" % (true_subgroup.mean(), len(targetX)))
    print("Detected %d" % (sum(detections_target)))
    print("Points detected from subgroup %d, proportion recalled %.2f" % (sum(intersecting_subgroup), sum(intersecting_subgroup)/sum(true_subgroup)))

    return sum(intersecting_subgroup)/sum(true_subgroup)

In [98]:
subgroup_recall = []
for i in range(num_jobs):
    explainer_data_jobid = explainer_data.replace('JOB', str(i+1))
    target_data_jobid = target_data.replace('JOB', str(i+1))
    subgroup_recall.append(
        get_subgroup_recall(explainer_data_jobid, target_data_jobid)
    )
subgroup_recall

X           0         1         2         3         4         5         6  \
0 -2.706037  1.148466  0.153934  2.691113 -0.996633 -1.474459  2.465269   
1 -1.627730  1.769245  1.762636  3.419146  0.100067 -0.809355 -1.090720   
2 -1.370093 -0.411300  2.972297  0.473433 -2.047570 -1.425986  1.250490   
3  2.490113  3.952222 -2.488247 -1.252834 -1.607532 -4.838166 -1.847584   
4 -2.246571  1.293351 -0.712542 -3.486282 -1.193299 -1.177189 -1.747765   

          7         8         9  
0 -3.582899 -1.386122 -0.522625  
1 -3.092955  1.964735 -2.202135  
2 -0.321027 -1.537673 -0.460061  
3 -2.047752  2.247956 -0.263828  
4  0.059428 -4.496516 -0.535524  
Y 0    0
1    1
2    0
3    1
4    0
Name: y, dtype: int64
Tree importance per feature
 [(9, 0.14443382543590422), (7, 0.14430040040177203), (0, 0.13620724004454304), (3, 0.11654583407896775), (8, 0.11466582018257775), (5, 0.10613053937043604), (1, 0.07326941987055295), (4, 0.07230294853703705), (6, 0.06452992366404746), (2, 0.02761404841416

[0.2857142857142857,
 0.2857142857142857,
 0.25,
 0.6666666666666666,
 0.625,
 0.8571428571428571,
 0.0,
 0.25,
 0.4444444444444444,
 0.5,
 0.3333333333333333,
 0.4,
 0.0,
 0.375,
 0.25,
 0.1111111111111111,
 0.5,
 0.5,
 1.0,
 0.3333333333333333,
 0.2,
 0.5,
 0.36363636363636365,
 0.7777777777777778,
 1.0,
 0.3,
 0.3333333333333333,
 0.16666666666666666,
 0.125,
 0.2857142857142857,
 0.42857142857142855,
 1.0,
 0.3333333333333333,
 0.4,
 0.5,
 0.375,
 0.3333333333333333,
 0.4,
 0.6,
 0.6666666666666666,
 0.4,
 0.14285714285714285,
 0.0,
 0.3333333333333333,
 0.16666666666666666,
 0.6666666666666666,
 0.25,
 0.6666666666666666,
 1.0,
 0.2857142857142857]

In [99]:
np.mean(subgroup_recall)

0.41936796536796533

In [89]:
alpha = 0.05
np.quantile(subgroup_recall, q=[alpha/2, 0.5, 1-alpha/2])

array([0.5467617 , 0.74917219, 0.92064601])

In [83]:
targetX, _ = read_csv(target_data.replace('JOB', '1'))
targetX = targetX.to_numpy()
detector = get_outcome_detector(explainer_data.replace('JOB', '1'))
detections_target = detector.predict(targetX) > 0
true_subgroup = (targetX[:,0] < -3.5) | (targetX[:,0] > 3.5)
intersecting_subgroup = true_subgroup & detections_target
print("True subgroup size %.2f from %d" % (true_subgroup.mean(), len(targetX)))
print("Detected %d" % (sum(detections_target)))
print("Points detected from subgroup %d, proportion recalled %.2f" % (sum(intersecting_subgroup), sum(intersecting_subgroup)/sum(true_subgroup)))

X           0         1         2         3         4         5         6  \
0  3.577257  0.873020  0.192995 -3.726985 -0.554776 -0.709518 -0.165483   
1 -2.627730  1.769245  1.762636  3.419146  0.100067 -0.809355 -1.090720   
2 -2.370093 -0.411300  2.972297  0.473433 -2.047570 -1.425986  1.250490   
3  1.490113  3.952222 -2.488247 -1.252834 -1.607532 -4.838166 -1.847584   
4 -3.246571  1.293351 -0.712542 -3.486282 -1.193299 -1.177189 -1.747765   

          7         8         9  
0 -1.254001 -0.087636 -0.954436  
1 -3.092955  1.964735 -2.202135  
2 -0.321027 -1.537673 -0.460061  
3 -2.047752  2.247956 -0.263828  
4  0.059428 -4.496516 -0.535524  
Y 0    0
1    0
2    1
3    0
4    0
Name: y, dtype: int64
Tree importance per feature
 [(0, 0.16827380704110398), (3, 0.1451756591385335), (5, 0.1274148122656654), (1, 0.12311368270341425), (4, 0.09804019426613579), (9, 0.08466011839926456), (8, 0.0771928702737456), (2, 0.07411360694647878), (6, 0.058985021568086146), (7, 0.0430302273975720